# Notebook 02: Modeling Pipeline
## Structural and Activity Distribution Shifts Contribute Distinctly to QSAR Generalization Failure in Natural Product Chemical Space

**Authors:** Abubakar Siddiq Salihu, Wan Mohd Nuzul Hakimi Wan Salleh  
**Corresponding author:** abusiddiq.salihu@umyu.edu.ng  
**Run environment:** Google Colab (CPU runtime)  
**Notebook 01 required:** Run `01_data_curation.ipynb` first and upload all CSV outputs to Google Drive before running this notebook.

---

### What this notebook does

This notebook implements the full modeling pipeline reported in the manuscript:

1. Mounts Google Drive and loads curated datasets from Notebook 01
2. Generates molecular descriptors (Morgan ECFP4, MACCS keys, RDKit 2D)
3. Computes Tanimoto similarity between NP test and synthetic training compounds
4. Generates UMAP chemical space projections (Figure 1) and Tanimoto distributions (Figure 2)
5. Implements three split strategies: random, scaffold (Bemis-Murcko), and NP holdout
6. Trains four ML models: Random Forest, XGBoost, SVM, feedforward neural network
7. Evaluates performance across all model-split-target combinations (Figure 3, Table 2)
8. Performs domain shift correlation analysis (Figure 4)
9. Runs SHAP interpretability analysis (Figure 5)
10. Generates supplementary figures S1 (pChEMBL distributions) and S2 (Tanimoto by criterion)
11. Exports all publication-ready figures and results tables

**Targets:** AChE (CHEMBL220), 5-LOX (CHEMBL215), COX-2 (CHEMBL230)  
**Primary descriptor:** Morgan fingerprints (ECFP4, radius 2, 2048 bits)  
**Primary metric:** Matthews Correlation Coefficient (MCC)  
**Estimated runtime:** 2-4 hours on Colab free CPU tier  

---

### Before running
- Ensure all CSV files from Notebook 01 are in `My Drive/np_qsar_project/data/`
- Use CPU runtime (Runtime > Change runtime type > CPU)
- Keep the Colab tab active to prevent session timeout
- Set `CONFIG['data_dir']` and `CONFIG['output_dir']` in Cell 01 to match your Drive paths

---

### Citation
If you use this code, please cite:  
Salihu AS, Wan Salleh WMNH. Structural and Activity Distribution Shifts Contribute Distinctly to QSAR Generalization Failure in Natural Product Chemical Space: A Systematic Benchmarking Study. *Journal of Cheminformatics* (under review).

---

## Cell 00: Mount Google Drive and Install Dependencies

In [ ]:
# ============================================================
# CELL 00: Mount Google Drive and Install Dependencies
# Run this cell first. It will prompt for Google account access.
# After mounting, verify the data path is correct.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys

# Install packages not available by default in Colab
packages = [
    'umap-learn',
    'shap',
    'xgboost',
]
for pkg in packages:
    print(f'Installing {pkg}...')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q']
    )

# Verify RDKit is available in Colab
try:
    from rdkit import Chem
    print('[OK] RDKit available')
except ImportError:
    print('Installing RDKit...')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', 'rdkit', '-q']
    )

print('\nAll dependencies ready.')

## Cell 01: Imports and Configuration

In [ ]:
# ============================================================
# CELL 01: Imports and Global Configuration
# ============================================================

import os
import warnings
import pickle
from pathlib import Path
from datetime import datetime
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm
from scipy import stats

# RDKit
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, MACCSkeys, Descriptors, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

# ML
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    matthews_corrcoef, roc_auc_score,
    average_precision_score, balanced_accuracy_score,
    confusion_matrix
)
import xgboost as xgb

# Dimensionality reduction
import umap

# SHAP
import shap

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore')

# ---- GLOBAL CONFIGURATION ----
CONFIG = {
    # Google Drive data path - matches Notebook 01 output location
    'data_dir': '/content/drive/MyDrive/np_qsar_project/data',

    # Output directory on Drive
    'output_dir': '/content/drive/MyDrive/np_qsar_project/results',

    # Targets - must match Notebook 01 file prefixes exactly
    'targets': ['AChE', '5LOX', 'COX2'],

    # Descriptor settings
    'morgan_radius': 2,
    'morgan_bits': 2048,

    # Model settings
    'n_cv_folds': 5,
    'seed': 42,

    # Tanimoto AD threshold
    # Compounds with max Tanimoto to training set below this
    # are flagged as outside applicability domain
    'ad_threshold': 0.4,

    # Figure settings
    'dpi': 300,
    'fig_format': 'png'
}

# Set seeds for full reproducibility
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])

# Create output directory
Path(CONFIG['output_dir']).mkdir(parents=True, exist_ok=True)

# Verify data directory and files
print('Verifying input files...')
missing = []
for target in CONFIG['targets']:
    for suffix in ['synthetic_train', 'np_test', 'combined']:
        fpath = Path(CONFIG['data_dir']) / f'{target}_{suffix}.csv'
        if fpath.exists():
            size_kb = fpath.stat().st_size / 1024
            print(f'  [OK] {fpath.name} ({size_kb:.0f} KB)')
        else:
            print(f'  [MISSING] {fpath.name}')
            missing.append(str(fpath))

if missing:
    print(f'\n[ERROR] {len(missing)} files missing.')
    print('Ensure all Notebook 01 outputs are in the data_dir.')
else:
    print(f'\n[OK] All input files found. Session: '
          f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Cell 02: Descriptor Generation

Three descriptor sets are generated for all compounds. Morgan fingerprints capture circular atom environments (ECFP4 equivalent). MACCS keys capture predefined structural keys. RDKit 2D descriptors capture physicochemical properties. All three are used in model training to assess whether descriptor choice interacts with domain shift.

In [ ]:
# ============================================================
# CELL 02: Descriptor Generation Functions
# ============================================================

def morgan_fp(smiles: str, radius: int = 2,
              n_bits: int = 2048) -> np.ndarray | None:
    """Generate Morgan fingerprint (ECFP4 equivalent) as numpy array."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol, radius, nBits=n_bits
    )
    arr = np.zeros(n_bits, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


def maccs_fp(smiles: str) -> np.ndarray | None:
    """Generate 167-bit MACCS keys fingerprint."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros(167, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


# RDKit 2D descriptor names (208 descriptors)
RDKIT_DESC_NAMES = [name for name, _ in Descriptors.descList]

def rdkit_2d(smiles: str) -> np.ndarray | None:
    """Generate 208 RDKit 2D physicochemical descriptors."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        vals = []
        for _, fn in Descriptors.descList:
            try:
                v = fn(mol)
                vals.append(float(v) if v is not None else np.nan)
            except Exception:
                vals.append(np.nan)
        return np.array(vals, dtype=np.float32)
    except Exception:
        return None


def generate_descriptors(df: pd.DataFrame,
                          smiles_col: str = 'std_smiles',
                          desc_type: str = 'morgan') -> np.ndarray:
    """
    Generate descriptor matrix for a DataFrame of compounds.

    Parameters
    ----------
    df : pd.DataFrame with SMILES column
    smiles_col : name of SMILES column
    desc_type : 'morgan', 'maccs', or 'rdkit2d'

    Returns
    -------
    np.ndarray of shape (n_compounds, n_features)
    Rows with failed descriptor generation are filled with zeros.
    """
    fn_map = {
        'morgan': lambda s: morgan_fp(
            s, CONFIG['morgan_radius'], CONFIG['morgan_bits']
        ),
        'maccs': maccs_fp,
        'rdkit2d': rdkit_2d
    }
    fn = fn_map[desc_type]

    results = []
    failed = 0
    for smi in tqdm(df[smiles_col], desc=f'{desc_type}', leave=False):
        arr = fn(smi)
        if arr is None:
            failed += 1
            # Determine feature size from first successful result
            if results:
                arr = np.zeros_like(results[0])
            else:
                results.append(None)
                continue
        results.append(arr)

    # Handle case where first result was None
    none_indices = [i for i, r in enumerate(results) if r is None]
    if none_indices and len(results) > max(none_indices):
        ref = next(r for r in results if r is not None)
        for i in none_indices:
            results[i] = np.zeros_like(ref)

    if failed > 0:
        print(f'    Warning: {failed} molecules failed {desc_type} '
              f'generation (filled with zeros)')

    return np.vstack(results)


# Quick test
test_df = pd.DataFrame({'std_smiles': [
    'CC(=O)Oc1ccccc1C(=O)O',
    'COc1ccc2c(c1)CCN(C)C1CC2COC1=O'
]})
for dtype in ['morgan', 'maccs', 'rdkit2d']:
    mat = generate_descriptors(test_df, desc_type=dtype)
    print(f'[OK] {dtype}: shape {mat.shape}')
print('\nDescriptor functions ready.')

## Cell 03: Load Data and Generate All Descriptors

This cell loads all curated CSVs and generates descriptor matrices. Results are checkpointed to Google Drive after each target to protect against Colab session timeouts.

In [ ]:
# ============================================================
# CELL 03: Load Data and Generate Descriptor Matrices
#
# Checkpoint strategy:
# After generating descriptors for each target, results are
# saved to Drive. If the session resets, re-running this cell
# will load existing checkpoints rather than recomputing.
# This is critical for Colab free tier stability.
# ============================================================

checkpoint_dir = Path(CONFIG['output_dir']) / 'checkpoints'
checkpoint_dir.mkdir(exist_ok=True)

# Storage for all target data
# Structure: data[target][split_type] = {'X': array, 'y': array, 'df': df}
data = {}

DESCRIPTOR_TYPES = ['morgan', 'maccs', 'rdkit2d']

for target in CONFIG['targets']:
    print(f'\n{"-" * 50}')
    print(f'Processing {target}...')

    # Check if checkpoint exists
    chk_path = checkpoint_dir / f'{target}_descriptors.pkl'
    if chk_path.exists():
        print(f'  Loading from checkpoint: {chk_path.name}')
        with open(chk_path, 'rb') as f:
            data[target] = pickle.load(f)
        print(f'  [OK] Loaded from checkpoint')
        continue

    # Load CSVs
    syn_df = pd.read_csv(
        Path(CONFIG['data_dir']) / f'{target}_synthetic_train.csv'
    )
    np_df = pd.read_csv(
        Path(CONFIG['data_dir']) / f'{target}_np_test.csv'
    )
    combined_df = pd.read_csv(
        Path(CONFIG['data_dir']) / f'{target}_combined.csv'
    )

    print(f'  Synthetic: {len(syn_df)} | NP: {len(np_df)} | '
          f'Combined: {len(combined_df)}')

    # Drop any rows with missing SMILES
    syn_df = syn_df.dropna(subset=['std_smiles']).reset_index(drop=True)
    np_df = np_df.dropna(subset=['std_smiles']).reset_index(drop=True)
    combined_df = combined_df.dropna(
        subset=['std_smiles']
    ).reset_index(drop=True)

    target_data = {
        'syn_df': syn_df,
        'np_df': np_df,
        'combined_df': combined_df,
        'descriptors': {}
    }

    # Generate descriptors for all three sets
    for desc_type in DESCRIPTOR_TYPES:
        print(f'  Generating {desc_type} descriptors...')
        target_data['descriptors'][desc_type] = {
            'syn': generate_descriptors(syn_df, desc_type=desc_type),
            'np': generate_descriptors(np_df, desc_type=desc_type),
            'combined': generate_descriptors(
                combined_df, desc_type=desc_type
            )
        }
        syn_shape = target_data['descriptors'][desc_type]['syn'].shape
        np_shape = target_data['descriptors'][desc_type]['np'].shape
        print(f'    Synthetic: {syn_shape} | NP: {np_shape}')

    # Save checkpoint
    with open(chk_path, 'wb') as f:
        pickle.dump(target_data, f)
    print(f'  [OK] Checkpoint saved: {chk_path.name}')

    data[target] = target_data

print(f'\n{"=" * 50}')
print('All descriptor matrices ready.')
print(f'Targets loaded: {list(data.keys())}')

## Cell 04: Tanimoto Similarity Analysis

For each NP test compound, we calculate its maximum Tanimoto similarity to the synthetic training set. This quantifies domain shift numerically and is the key independent variable in the domain shift correlation analysis (Figure 4).

In [ ]:
# ============================================================
# CELL 04: Tanimoto Similarity Analysis
#
# Maximum Tanimoto similarity to training set is calculated
# for each NP test compound using Morgan fingerprints.
# Morgan FP is used here (not MACCS or 2D) because Tanimoto
# similarity on Morgan FP is the standard applicability domain
# metric in the QSAR literature (Tropsha et al.).
# ============================================================

def compute_max_tanimoto(query_smiles_list: list,
                          reference_smiles_list: list,
                          radius: int = 2,
                          n_bits: int = 2048,
                          batch_size: int = 500) -> np.ndarray:
    """
    Compute maximum Tanimoto similarity between each query compound
    and the reference set.

    Uses batched processing to manage memory on Colab.
    Returns array of shape (n_query,) with max Tanimoto per compound.
    """
    # Generate reference fingerprints
    ref_fps = []
    for smi in reference_smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            ref_fps.append(
                AllChem.GetMorganFingerprintAsBitVect(mol, radius, n_bits)
            )
    print(f'  Reference FPs generated: {len(ref_fps)}')

    max_similarities = []

    for i in tqdm(range(0, len(query_smiles_list), batch_size),
                  desc='Tanimoto'):
        batch = query_smiles_list[i:i + batch_size]
        for smi in batch:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                max_similarities.append(0.0)
                continue
            query_fp = AllChem.GetMorganFingerprintAsBitVect(
                mol, radius, n_bits
            )
            sims = DataStructs.BulkTanimotoSimilarity(query_fp, ref_fps)
            max_similarities.append(max(sims) if sims else 0.0)

    return np.array(max_similarities)


# Compute Tanimoto similarities for all targets
print('Computing Tanimoto similarities (NP test vs synthetic train)...')
print('This may take 5-15 minutes depending on dataset size.')

tanimoto_results = {}

for target in CONFIG['targets']:
    chk_path = checkpoint_dir / f'{target}_tanimoto.npy'

    if chk_path.exists():
        print(f'\n{target}: Loading Tanimoto from checkpoint')
        tanimoto_results[target] = np.load(str(chk_path))
        print(f'  Loaded {len(tanimoto_results[target])} values')
        continue

    print(f'\n{target}:')
    syn_df = data[target]['syn_df']
    np_df = data[target]['np_df']

    max_tan = compute_max_tanimoto(
        np_df['std_smiles'].tolist(),
        syn_df['std_smiles'].tolist()
    )

    np.save(str(chk_path), max_tan)
    tanimoto_results[target] = max_tan

    # Summary statistics
    print(f'  Max Tanimoto stats:')
    print(f'    Mean:   {max_tan.mean():.3f}')
    print(f'    Median: {np.median(max_tan):.3f}')
    print(f'    Min:    {max_tan.min():.3f}')
    print(f'    Max:    {max_tan.max():.3f}')
    ad_coverage = (max_tan >= CONFIG['ad_threshold']).mean() * 100
    print(f'    Within AD (>={CONFIG["ad_threshold"]}): '
          f'{ad_coverage:.1f}%')

print('\n[OK] Tanimoto analysis complete.')

## Cell 05: Figure 1 and 2 - Chemical Space Analysis

UMAP visualization of NP vs synthetic chemical space, and Tanimoto similarity distributions. These are Figures 1 and 2 in the paper.

In [ ]:
# ============================================================
# CELL 05: Chemical Space Analysis - Figures 1 and 2
#
# Figure 1: UMAP projection showing NP vs synthetic separation
# Figure 2: Tanimoto similarity distributions
#
# UMAP is run on Morgan fingerprints (2048-bit).
# For computational efficiency on Colab, we subsample the
# synthetic set to max 2000 compounds for UMAP visualization
# only. Full dataset is used for all model training.
# ============================================================

COLORS = {
    'synthetic': '#4472C4',
    'natural_product': '#2E8B57',
    'active': '#D62728',
    'inactive': '#7F7F7F'
}

# ---- FIGURE 1: UMAP Chemical Space ----
print('Generating Figure 1: UMAP chemical space...')
print('UMAP runtime: ~5-10 minutes on Colab CPU')

fig1, axes1 = plt.subplots(1, 3, figsize=(18, 6))
fig1.suptitle(
    'Figure 1. Chemical Space Distribution: Synthetic vs '
    'Natural Product Compounds',
    fontsize=13, fontweight='bold', y=1.02
)

umap_results = {}

for idx, target in enumerate(CONFIG['targets']):
    ax = axes1[idx]
    syn_df = data[target]['syn_df']
    np_df = data[target]['np_df']

    # Subsample synthetic for UMAP speed (visualization only)
    MAX_SYN_UMAP = 2000
    if len(syn_df) > MAX_SYN_UMAP:
        syn_sample = syn_df.sample(
            MAX_SYN_UMAP, random_state=CONFIG['seed']
        )
    else:
        syn_sample = syn_df.copy()

    # Combine for joint UMAP
    combined_sample = pd.concat(
        [syn_sample, np_df], ignore_index=True
    )
    labels = (['synthetic'] * len(syn_sample) +
              ['natural_product'] * len(np_df))

    # Get Morgan FP matrix
    print(f'  {target}: Generating UMAP descriptors...')
    X_umap = generate_descriptors(combined_sample, desc_type='morgan')

    # Fit UMAP
    print(f'  {target}: Fitting UMAP on {len(combined_sample)} compounds...')
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        metric='jaccard',  # Jaccard is appropriate for binary fingerprints
        random_state=CONFIG['seed'],
        low_memory=True    # Essential for Colab memory management
    )
    embedding = reducer.fit_transform(X_umap)
    umap_results[target] = {
        'embedding': embedding,
        'labels': labels
    }

    # Plot
    for compound_type, color in [
        ('synthetic', COLORS['synthetic']),
        ('natural_product', COLORS['natural_product'])
    ]:
        mask = [l == compound_type for l in labels]
        ax.scatter(
            embedding[mask, 0], embedding[mask, 1],
            c=color, alpha=0.4, s=8,
            label=compound_type.replace('_', ' ').title()
        )

    ax.set_title(f'{target}\n(Syn: {len(syn_sample)}, NP: {len(np_df)})',
                 fontsize=11)
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')
    ax.legend(markerscale=2, fontsize=9)
    ax.tick_params(labelsize=8)
    print(f'  {target}: UMAP complete')

plt.tight_layout()
fig1_path = Path(CONFIG['output_dir']) / 'Figure1_UMAP_chemical_space.png'
fig1.savefig(str(fig1_path), dpi=CONFIG['dpi'], bbox_inches='tight')
plt.show()
print(f'Figure 1 saved: {fig1_path.name}')

# ---- FIGURE 2: Tanimoto Similarity Distributions ----
print('\nGenerating Figure 2: Tanimoto distributions...')

fig2, axes2 = plt.subplots(1, 3, figsize=(15, 5))
fig2.suptitle(
    'Figure 2. Maximum Tanimoto Similarity of NP Test Compounds '
    'to Synthetic Training Set',
    fontsize=13, fontweight='bold'
)

for idx, target in enumerate(CONFIG['targets']):
    ax = axes2[idx]
    max_tan = tanimoto_results[target]

    ax.hist(max_tan, bins=30, color=COLORS['natural_product'],
            edgecolor='white', alpha=0.85, density=True)
    ax.axvline(CONFIG['ad_threshold'], color='red', linestyle='--',
               linewidth=1.5,
               label=f'AD threshold ({CONFIG["ad_threshold"]})')
    ax.axvline(np.median(max_tan), color='orange', linestyle='-.',
               linewidth=1.5,
               label=f'Median ({np.median(max_tan):.2f})')

    within_ad = (max_tan >= CONFIG['ad_threshold']).mean() * 100
    ax.set_title(f'{target}\nWithin AD: {within_ad:.1f}%', fontsize=11)
    ax.set_xlabel('Max Tanimoto Similarity to Training Set')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.tight_layout()
fig2_path = Path(CONFIG['output_dir']) / 'Figure2_Tanimoto_distributions.png'
fig2.savefig(str(fig2_path), dpi=CONFIG['dpi'], bbox_inches='tight')
plt.show()
print(f'Figure 2 saved: {fig2_path.name}')

## Cell 06: Split Strategy Implementation

Three split strategies are applied to each target. Random split is the standard but methodologically weak approach. Scaffold split is more realistic. NP holdout is the novel evaluation in this paper.

In [ ]:
# ============================================================
# CELL 06: Split Strategy Implementation
#
# Split 1: Random split (80/20) on combined dataset
#   - Standard approach in most published QSAR papers
#   - Compounds from same scaffold can appear in train and test
#   - Expected to give highest (most optimistic) performance
#
# Split 2: Scaffold split on combined dataset
#   - Bemis-Murcko scaffolds assigned in Notebook 01
#   - All compounds from a scaffold go entirely to train or test
#   - More realistic estimate of generalization performance
#
# Split 3: NP holdout
#   - Train: synthetic ChEMBL compounds only
#   - Test: natural product compounds only
#   - Simulates real-world scenario of applying a QSAR model
#     trained on synthetic screening data to NP scaffolds
#   - This is the key evaluation in this paper
# ============================================================

def random_split(combined_df: pd.DataFrame,
                  test_size: float = 0.2,
                  seed: int = 42) -> tuple:
    """Standard random train/test split."""
    train_idx, test_idx = train_test_split(
        np.arange(len(combined_df)),
        test_size=test_size,
        stratify=combined_df['activity'].values,
        random_state=seed
    )
    return train_idx, test_idx


def scaffold_split(combined_df: pd.DataFrame,
                    test_size: float = 0.2,
                    seed: int = 42) -> tuple:
    """
    Scaffold-based split using Bemis-Murcko scaffolds.

    Scaffolds are sorted by size (largest first) and assigned
    to test until test_size fraction is reached. This avoids
    structurally similar compounds appearing in both splits.
    """
    scaffold_col = 'scaffold'
    if scaffold_col not in combined_df.columns:
        print('  Warning: scaffold column missing, falling back to random split')
        return random_split(combined_df, test_size, seed)

    # Group by scaffold
    scaffold_to_indices = {}
    for idx, row in combined_df.iterrows():
        sc = row[scaffold_col] if pd.notna(row[scaffold_col]) else 'no_scaffold'
        if sc not in scaffold_to_indices:
            scaffold_to_indices[sc] = []
        scaffold_to_indices[sc].append(idx)

    # Sort scaffolds by size (largest first)
    scaffolds_sorted = sorted(
        scaffold_to_indices.items(),
        key=lambda x: len(x[1]),
        reverse=True
    )

    # Assign to test until target size reached
    target_test_size = int(len(combined_df) * test_size)
    test_indices = []
    train_indices = []

    rng = np.random.RandomState(seed)
    scaffold_list = [(sc, idxs) for sc, idxs in scaffolds_sorted]
    rng.shuffle(scaffold_list)

    for sc, idxs in scaffold_list:
        if len(test_indices) < target_test_size:
            test_indices.extend(idxs)
        else:
            train_indices.extend(idxs)

    # Convert to positional indices
    all_idx = combined_df.index.tolist()
    idx_map = {v: i for i, v in enumerate(all_idx)}
    train_pos = [idx_map[i] for i in train_indices if i in idx_map]
    test_pos = [idx_map[i] for i in test_indices if i in idx_map]

    return np.array(train_pos), np.array(test_pos)


def np_holdout_split(combined_df: pd.DataFrame) -> tuple:
    """
    NP holdout split: train on synthetic, test on NP.
    compound_type column must be present (set in Notebook 01).
    """
    combined_df = combined_df.reset_index(drop=True)
    train_idx = combined_df.index[
        combined_df['compound_type'] == 'synthetic'
    ].tolist()
    test_idx = combined_df.index[
        combined_df['compound_type'] == 'natural_product'
    ].tolist()
    return np.array(train_idx), np.array(test_idx)


# Generate and store splits for all targets
print('Generating split indices for all targets...')

splits = {}
for target in CONFIG['targets']:
    combined_df = data[target]['combined_df'].reset_index(drop=True)

    rand_train, rand_test = random_split(combined_df, seed=CONFIG['seed'])
    scaf_train, scaf_test = scaffold_split(combined_df, seed=CONFIG['seed'])
    np_train, np_test = np_holdout_split(combined_df)

    splits[target] = {
        'random':   (rand_train, rand_test),
        'scaffold': (scaf_train, scaf_test),
        'np_holdout': (np_train, np_test)
    }

    print(f'\n{target}:')
    for split_name, (tr, te) in splits[target].items():
        y_all = combined_df['activity'].values
        train_active = y_all[tr].sum()
        test_active = y_all[te].sum()
        print(f'  {split_name:12s}: '
              f'train={len(tr)} (active={train_active}), '
              f'test={len(te)} (active={test_active})')

print('\n[OK] Splits generated.')

## Cell 07: Neural Network Architecture

In [ ]:
# ============================================================
# CELL 07: Feedforward Neural Network
#
# Simple 3-layer FFNN for binary classification.
# Architecture kept minimal for CPU execution on Colab.
# Batch normalization and dropout included for regularization.
# Early stopping prevents overfitting on small NP test sets.
# ============================================================

class FeedForwardNet(nn.Module):
    """
    3-layer feedforward neural network for binary QSAR classification.

    Architecture:
    Input -> Dense(512) -> BN -> ReLU -> Dropout(0.3)
          -> Dense(256) -> BN -> ReLU -> Dropout(0.3)
          -> Dense(128) -> BN -> ReLU -> Dropout(0.2)
          -> Dense(1) -> Sigmoid
    """
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.net(x)


def train_nn(X_train: np.ndarray, y_train: np.ndarray,
             X_val: np.ndarray, y_val: np.ndarray,
             max_epochs: int = 100,
             batch_size: int = 64,
             patience: int = 10,
             seed: int = 42) -> FeedForwardNet:
    """
    Train FFNN with early stopping based on validation loss.
    Returns trained model in eval mode.
    """
    torch.manual_seed(seed)

    # Normalize input (important for NN, not for tree models)
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train.astype(np.float32))
    X_val_sc = scaler.transform(X_val.astype(np.float32))

    # Create datasets
    train_ds = TensorDataset(
        torch.FloatTensor(X_train_sc),
        torch.FloatTensor(y_train.astype(np.float32))
    )
    val_ds = TensorDataset(
        torch.FloatTensor(X_val_sc),
        torch.FloatTensor(y_val.astype(np.float32))
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

    model = FeedForwardNet(X_train.shape[1])
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    # Class weight for imbalanced data
    pos_weight = torch.tensor(
        [(y_train == 0).sum() / max((y_train == 1).sum(), 1)],
        dtype=torch.float32
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # Early stopping
    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None

    for epoch in range(max_epochs):
        # Training
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                val_loss += criterion(
                    model(X_batch).squeeze(), y_batch
                ).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in
                          model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_state:
        model.load_state_dict(best_state)

    # Store scaler on model for prediction use
    model.scaler = scaler
    model.eval()
    return model


def predict_nn(model: FeedForwardNet,
               X_test: np.ndarray) -> np.ndarray:
    """Generate probability predictions from trained FFNN."""
    X_sc = model.scaler.transform(X_test.astype(np.float32))
    with torch.no_grad():
        logits = model(torch.FloatTensor(X_sc)).squeeze()
        probs = torch.sigmoid(logits).numpy()
    return probs


print('[OK] Neural network architecture defined.')

## Cell 08: Model Training and Evaluation

This is the core computational cell. Four models x three splits x three targets = 36 evaluations. Morgan fingerprints are used as the primary descriptor for comparability. Runtime: approximately 60-90 minutes on Colab CPU.

In [ ]:
# ============================================================
# CELL 08: Model Training and Evaluation
#
# Primary descriptor: Morgan fingerprints
# Models: Random Forest, XGBoost, SVM, FFNN
# Splits: random, scaffold, np_holdout
# Metrics: MCC (primary), AUROC, AUPRC, balanced accuracy
#
# MCC is chosen as the primary metric because:
# 1. It accounts for all four cells of the confusion matrix
# 2. It is robust to class imbalance (unlike accuracy)
# 3. It is increasingly used as primary metric in
#    benchmarking papers (Chicco & Jurman, 2020, BMC Genomics)
# ============================================================

def compute_metrics(y_true: np.ndarray,
                     y_prob: np.ndarray,
                     threshold: float = 0.5) -> dict:
    """
    Compute all evaluation metrics from probability predictions.
    Returns dict with MCC, AUROC, AUPRC, balanced accuracy.
    """
    y_pred = (y_prob >= threshold).astype(int)

    # Handle edge cases where only one class is present in y_true
    if len(np.unique(y_true)) < 2:
        return {
            'MCC': np.nan, 'AUROC': np.nan,
            'AUPRC': np.nan, 'BalAcc': np.nan
        }

    return {
        'MCC': matthews_corrcoef(y_true, y_pred),
        'AUROC': roc_auc_score(y_true, y_prob),
        'AUPRC': average_precision_score(y_true, y_prob),
        'BalAcc': balanced_accuracy_score(y_true, y_pred)
    }


def get_model_definitions(seed: int = 42) -> dict:
    """Return dict of model name to initialized model object."""
    return {
        'RandomForest': RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            min_samples_leaf=2,
            class_weight='balanced',
            n_jobs=-1,
            random_state=seed
        ),
        'XGBoost': xgb.XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            use_label_encoder=False,
            eval_metric='logloss',
            random_state=seed,
            verbosity=0
        ),
        'SVM': Pipeline([
            ('scaler', StandardScaler()),
            ('svm', SVC(
                kernel='rbf',
                C=1.0,
                probability=True,
                class_weight='balanced',
                random_state=seed
            ))
        ]),
        # FFNN handled separately via train_nn / predict_nn
    }


# ---- Main training loop ----
results_records = []
trained_models = {}  # Store for SHAP analysis

total_runs = len(CONFIG['targets']) * 4 * 3  # targets x models x splits
run_count = 0

print(f'Starting model training: {total_runs} total evaluations')
print(f'Estimated runtime: 60-90 minutes')
print('=' * 60)

for target in CONFIG['targets']:
    combined_df = data[target]['combined_df'].reset_index(drop=True)
    X_all = data[target]['descriptors']['morgan']['combined']
    y_all = combined_df['activity'].values

    trained_models[target] = {}

    for split_name, (train_idx, test_idx) in splits[target].items():
        X_train = X_all[train_idx]
        X_test = X_all[test_idx]
        y_train = y_all[train_idx]
        y_test = y_all[test_idx]

        # For NP validation split within training (for FFNN early stopping)
        if len(X_train) > 100:
            val_size = min(0.1, 200 / len(X_train))
            X_tr, X_val, y_tr, y_val = train_test_split(
                X_train, y_train,
                test_size=val_size,
                stratify=y_train,
                random_state=CONFIG['seed']
            )
        else:
            X_tr, X_val, y_tr, y_val = X_train, X_train, y_train, y_train

        # Skip if test set is too small or has only one class
        if len(y_test) < 10 or len(np.unique(y_test)) < 2:
            print(f'  [SKIP] {target}/{split_name}: '
                  f'test set too small or single class')
            continue

        # ---- Train sklearn models ----
        for model_name, model in get_model_definitions(
            CONFIG['seed']
        ).items():
            run_count += 1
            print(f'[{run_count}/{total_runs}] '
                  f'{target} | {split_name} | {model_name}...',
                  end=' ')

            try:
                model.fit(X_train, y_train)
                y_prob = model.predict_proba(X_test)[:, 1]
                metrics = compute_metrics(y_test, y_prob)

                # Store RF and XGBoost for SHAP and NP subgroup decomposition analysis
                if split_name == 'np_holdout' and model_name in [
                    'RandomForest', 'XGBoost'
                ]:
                    trained_models[target][model_name] = {
                        'model': model,
                        'X_train': X_train,
                        'X_test_np': X_test,
                        'y_test_np': y_test,
                        'split': split_name
                    }
                # Also store random split RF for SHAP comparison
                if split_name == 'random' and model_name == 'RandomForest':
                    if 'RandomForest_random' not in trained_models[target]:
                        trained_models[target]['RandomForest_random'] = {
                            'model': model,
                            'X_train': X_train,
                            'X_test': X_test
                        }

                print(f"MCC={metrics['MCC']:.3f} "
                      f"AUROC={metrics['AUROC']:.3f}")

            except Exception as e:
                metrics = {
                    'MCC': np.nan, 'AUROC': np.nan,
                    'AUPRC': np.nan, 'BalAcc': np.nan
                }
                print(f'FAILED: {e}')

            results_records.append({
                'Target': target,
                'Split': split_name,
                'Model': model_name,
                'n_train': len(X_train),
                'n_test': len(X_test),
                **metrics
            })

        # ---- Train FFNN ----
        run_count += 1
        print(f'[{run_count}/{total_runs}] '
              f'{target} | {split_name} | FFNN...',
              end=' ')
        try:
            nn_model = train_nn(
                X_tr, y_tr, X_val, y_val,
                seed=CONFIG['seed']
            )
            y_prob_nn = predict_nn(nn_model, X_test)
            metrics_nn = compute_metrics(y_test, y_prob_nn)
            print(f"MCC={metrics_nn['MCC']:.3f} "
                  f"AUROC={metrics_nn['AUROC']:.3f}")
        except Exception as e:
            metrics_nn = {
                'MCC': np.nan, 'AUROC': np.nan,
                'AUPRC': np.nan, 'BalAcc': np.nan
            }
            print(f'FAILED: {e}')

        results_records.append({
            'Target': target,
            'Split': split_name,
            'Model': 'FFNN',
            'n_train': len(X_train),
            'n_test': len(X_test),
            **metrics_nn
        })

# Convert to DataFrame
results_df = pd.DataFrame(results_records)
results_df.to_csv(
    Path(CONFIG['output_dir']) / 'model_results.csv', index=False
)
print('\n' + '=' * 60)
print('Training complete. Results saved.')
print(results_df.groupby(['Target', 'Split'])['MCC'].mean().round(3)
      .to_string())

## Cell 09: Figure 3 - Performance Heatmap

In [ ]:
# ============================================================
# CELL 09: Figure 3 - Performance Heatmap
#
# Heatmap showing MCC values across all model x split x target
# combinations. This is the central results figure.
# ============================================================

print('Generating Figure 3: Performance heatmap...')

split_order = ['random', 'scaffold', 'np_holdout']
split_labels = {
    'random': 'Random\nSplit',
    'scaffold': 'Scaffold\nSplit',
    'np_holdout': 'NP\nHoldout'
}
model_order = ['RandomForest', 'XGBoost', 'SVM', 'FFNN']

fig3, axes3 = plt.subplots(1, 3, figsize=(18, 6))
fig3.suptitle(
    'Figure 3. Model Performance (MCC) Across Split Strategies and Targets',
    fontsize=13, fontweight='bold'
)

for idx, target in enumerate(CONFIG['targets']):
    ax = axes3[idx]
    target_df = results_df[results_df['Target'] == target]

    # Build matrix: rows = models, cols = splits
    matrix = np.full((len(model_order), len(split_order)), np.nan)
    for r, model in enumerate(model_order):
        for c, split in enumerate(split_order):
            row = target_df[
                (target_df['Model'] == model) &
                (target_df['Split'] == split)
            ]
            if len(row) > 0:
                matrix[r, c] = row['MCC'].values[0]

    # Plot heatmap
    im = ax.imshow(matrix, cmap='RdYlGn', vmin=-0.1, vmax=0.9,
                   aspect='auto')

    # Annotations
    for r in range(len(model_order)):
        for c in range(len(split_order)):
            val = matrix[r, c]
            if not np.isnan(val):
                text_color = 'white' if val < 0.3 or val > 0.7 else 'black'
                ax.text(c, r, f'{val:.2f}', ha='center', va='center',
                        fontsize=11, fontweight='bold', color=text_color)

    ax.set_xticks(range(len(split_order)))
    ax.set_xticklabels([split_labels[s] for s in split_order], fontsize=10)
    ax.set_yticks(range(len(model_order)))
    ax.set_yticklabels(model_order, fontsize=10)
    ax.set_title(f'{target}', fontsize=12, fontweight='bold')

    plt.colorbar(im, ax=ax, label='MCC', shrink=0.8)

plt.tight_layout()
fig3_path = Path(CONFIG['output_dir']) / 'Figure3_performance_heatmap.png'
fig3.savefig(str(fig3_path), dpi=CONFIG['dpi'], bbox_inches='tight')
plt.show()
print(f'Figure 3 saved: {fig3_path.name}')

## Cell 10: Figure 4 - Domain Shift Correlation

For each NP test compound, we plot binary prediction correctness against maximum Tanimoto similarity
to the synthetic training set. This tests whether Tanimoto-based applicability domain metrics
predict model failure in NP chemical space.

Key finding: the relationship is target-dependent. AChE shows a significant correlation
(Spearman r = 0.196, p < 0.001). 5-LOX and COX-2 show no significant binary correctness correlation
despite significant confidence-Tanimoto correlations, consistent with the two-mechanism framework
described in the manuscript (structural distance predicts confidence universally, but activity
distribution mismatch drives MCC degradation independently for COCONUT-confirmed NP subgroups).

In [ ]:
# ============================================================
# CELL 10: Figure 4 - Domain Shift Correlation
#
# For each NP test compound, we have:
# - Maximum Tanimoto similarity to training set (x-axis)
# - Whether the model prediction was correct (used to compute
#   a sliding window accuracy as a function of Tanimoto)
#
# Spearman correlation between Tanimoto and prediction
# correctness quantifies the domain shift effect.
#
# We use RandomForest (np_holdout split) for this analysis
# as it is the most interpretable model.
# ============================================================

print('Generating Figure 4: Domain shift correlation...')

fig4, axes4 = plt.subplots(1, 3, figsize=(18, 6))
fig4.suptitle(
    'Figure 4. Prediction Accuracy vs Tanimoto Similarity to Training Set\n'
    '(Random Forest, NP Holdout Split)',
    fontsize=13, fontweight='bold'
)

correlation_summary = []

for idx, target in enumerate(CONFIG['targets']):
    ax = axes4[idx]

    # Get RF model trained on NP holdout split
    if ('RandomForest' not in trained_models.get(target, {}) or
            'X_test_np' not in trained_models[target]['RandomForest']):
        ax.text(0.5, 0.5, 'Model not available',
                transform=ax.transAxes, ha='center')
        continue

    model_data = trained_models[target]['RandomForest']
    rf_model = model_data['model']
    X_test_np = model_data['X_test_np']
    y_test_np = model_data['y_test_np']

    # Get predictions
    y_prob_np = rf_model.predict_proba(X_test_np)[:, 1]
    y_pred_np = (y_prob_np >= 0.5).astype(int)
    correct = (y_pred_np == y_test_np).astype(int)

    # Tanimoto similarities
    max_tan = tanimoto_results[target]

    # Align lengths (should match after consistent processing)
    min_len = min(len(correct), len(max_tan))
    correct = correct[:min_len]
    max_tan = max_tan[:min_len]
    y_prob_np = y_prob_np[:min_len]

    # Spearman correlation
    spearman_r, spearman_p = stats.spearmanr(max_tan, correct)

    # Bin compounds by Tanimoto for visualization
    n_bins = 8
    bin_edges = np.percentile(max_tan, np.linspace(0, 100, n_bins + 1))
    bin_edges = np.unique(bin_edges)  # Remove duplicate edges
    bin_centers = []
    bin_accuracies = []
    bin_sizes = []

    for i in range(len(bin_edges) - 1):
        mask = (max_tan >= bin_edges[i]) & (max_tan < bin_edges[i + 1])
        if mask.sum() >= 5:  # Minimum bin size
            bin_centers.append((bin_edges[i] + bin_edges[i + 1]) / 2)
            bin_accuracies.append(correct[mask].mean())
            bin_sizes.append(mask.sum())

    # Scatter: individual compounds
    ax.scatter(max_tan, correct + np.random.normal(0, 0.02, len(correct)),
               c=correct, cmap='RdYlGn', alpha=0.2, s=15,
               vmin=0, vmax=1)

    # Binned accuracy line
    if bin_centers:
        ax.plot(bin_centers, bin_accuracies, 'b-o',
                linewidth=2, markersize=6,
                label='Binned accuracy', zorder=5)

    ax.axvline(CONFIG['ad_threshold'], color='red', linestyle='--',
               alpha=0.7, label=f'AD threshold')
    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5,
               label='Random baseline')

    p_str = f'p={spearman_p:.3f}' if spearman_p >= 0.001 else 'p<0.001'
    ax.set_title(
        f'{target}\nSpearman r={spearman_r:.3f}, {p_str}',
        fontsize=11
    )
    ax.set_xlabel('Max Tanimoto Similarity to Training Set')
    ax.set_ylabel('Prediction Correct (1=Yes, 0=No)')
    ax.set_ylim(-0.15, 1.15)
    ax.legend(fontsize=8)

    correlation_summary.append({
        'Target': target,
        'Spearman_r': round(spearman_r, 4),
        'Spearman_p': round(spearman_p, 4),
        'N_NP_test': min_len
    })

plt.tight_layout()
fig4_path = Path(CONFIG['output_dir']) / 'Figure4_domain_shift_correlation.png'
fig4.savefig(str(fig4_path), dpi=CONFIG['dpi'], bbox_inches='tight')
plt.show()

corr_df = pd.DataFrame(correlation_summary)
corr_df.to_csv(
    Path(CONFIG['output_dir']) / 'correlation_summary.csv', index=False
)
print('\nCorrelation Summary:')
print(corr_df.to_string(index=False))
print(f'\nFigure 4 saved: {fig4_path.name}')

## Cell 11: Figure 5 - SHAP Interpretability Analysis

SHAP feature importances are compared between synthetic test compounds and NP test compounds for the same trained model. Divergence in feature importance profiles is mechanistic evidence for why models fail on NP scaffolds.

In [ ]:
# ============================================================
# CELL 11: Figure 5 - SHAP Analysis
#
# We compare SHAP value distributions for:
# A) RandomForest trained on synthetic, evaluated on synthetic test
# B) Same model evaluated on NP test compounds
#
# If the top features driving predictions shift between A and B,
# that is evidence the model is relying on structural patterns
# present in synthetic compounds but absent in NPs.
#
# Morgan bit positions are not chemically interpretable alone,
# so we report top-20 features by mean |SHAP| and note that
# the divergence pattern (not individual bits) is the finding.
# ============================================================

print('Generating Figure 5: SHAP analysis...')
print('SHAP runtime: ~10-20 minutes on Colab CPU')

SHAP_SAMPLE_SIZE = 200  # Subsample for SHAP speed on Colab

fig5, axes5 = plt.subplots(3, 2, figsize=(16, 18))
fig5.suptitle(
    'Figure 5. SHAP Feature Importance: Synthetic vs '
    'Natural Product Test Compounds\n(Random Forest, NP Holdout Split)',
    fontsize=13, fontweight='bold'
)

for idx, target in enumerate(CONFIG['targets']):
    ax_syn = axes5[idx, 0]
    ax_np = axes5[idx, 1]

    if 'RandomForest' not in trained_models.get(target, {}):
        for ax in [ax_syn, ax_np]:
            ax.text(0.5, 0.5, 'Model not available',
                    transform=ax.transAxes, ha='center')
        continue

    model_data = trained_models[target]['RandomForest']
    rf_model = model_data['model']
    X_train = model_data['X_train']
    X_test_np = model_data['X_test_np']

    # Get synthetic test set from random split for comparison
    combined_df = data[target]['combined_df'].reset_index(drop=True)
    X_all = data[target]['descriptors']['morgan']['combined']
    rand_train_idx, rand_test_idx = splits[target]['random']
    X_test_syn = X_all[rand_test_idx]

    # Subsample for SHAP
    rng = np.random.RandomState(CONFIG['seed'])

    n_syn = min(SHAP_SAMPLE_SIZE, len(X_test_syn))
    syn_sample_idx = rng.choice(len(X_test_syn), n_syn, replace=False)
    X_syn_sample = X_test_syn[syn_sample_idx]

    n_np = min(SHAP_SAMPLE_SIZE, len(X_test_np))
    np_sample_idx = rng.choice(len(X_test_np), n_np, replace=False)
    X_np_sample = X_test_np[np_sample_idx]

    # SHAP TreeExplainer (efficient for RF)
    print(f'  {target}: Computing SHAP values...')
    explainer = shap.TreeExplainer(rf_model)

    shap_syn = explainer.shap_values(X_syn_sample)
    shap_np = explainer.shap_values(X_np_sample)

    # For binary classification, shap_values returns list of two arrays
    # Take values for class 1 (active)
    if isinstance(shap_syn, list):
        shap_syn = shap_syn[1]
    if isinstance(shap_np, list):
        shap_np = shap_np[1]

    # Mean absolute SHAP per feature
    mean_shap_syn = np.abs(shap_syn).mean(axis=0)
    mean_shap_np = np.abs(shap_np).mean(axis=0)

    # Top 20 features by synthetic SHAP importance
    top20_syn_idx = np.argsort(mean_shap_syn)[::-1][:20]
    top20_np_idx = np.argsort(mean_shap_np)[::-1][:20]

    # Plot synthetic SHAP
    ax_syn.barh(
        range(20),
        mean_shap_syn[top20_syn_idx][::-1],
        color=COLORS['synthetic'], alpha=0.8
    )
    ax_syn.set_yticks(range(20))
    ax_syn.set_yticklabels(
        [f'Bit {i}' for i in top20_syn_idx[::-1]], fontsize=8
    )
    ax_syn.set_xlabel('Mean |SHAP value|')
    ax_syn.set_title(
        f'{target} - Synthetic Test\n(n={n_syn})', fontsize=10
    )

    # Plot NP SHAP
    ax_np.barh(
        range(20),
        mean_shap_np[top20_np_idx][::-1],
        color=COLORS['natural_product'], alpha=0.8
    )
    ax_np.set_yticks(range(20))
    ax_np.set_yticklabels(
        [f'Bit {i}' for i in top20_np_idx[::-1]], fontsize=8
    )
    ax_np.set_xlabel('Mean |SHAP value|')
    ax_np.set_title(
        f'{target} - NP Test\n(n={n_np})', fontsize=10
    )

    # Overlap score: fraction of top-20 features shared
    overlap = len(set(top20_syn_idx) & set(top20_np_idx))
    print(f'  {target}: Top-20 feature overlap '
          f'(syn vs NP): {overlap}/20 ({overlap*5}%)')

plt.tight_layout()
fig5_path = Path(CONFIG['output_dir']) / 'Figure5_SHAP_comparison.png'
fig5.savefig(str(fig5_path), dpi=CONFIG['dpi'], bbox_inches='tight')
plt.show()
print(f'\nFigure 5 saved: {fig5_path.name}')

## Cell 10b: Supplementary Figures S1 and S2

**Figure S1:** pChEMBL value distributions by compound group per target.  
Shows that COCONUT-confirmed NPs cluster below the activity threshold (pChEMBL 6.0) for all three targets,
while NP-score-only compounds more closely match the synthetic training distribution for AChE and 5-LOX.
This figure supports the activity distribution mismatch mechanism identified in Section 3.3.

**Figure S2:** Tanimoto similarity distributions by NP classification criterion per target.  
Demonstrates structural equidistance between COCONUT-confirmed and NP-score-only subgroups for 5-LOX (p = 0.194)
and COX-2 (p = 0.999), and the AChE confounding case (p = 0.002). This is the key evidence that the
performance divergence between NP subgroups cannot be attributed to structural distance alone.

In [ ]:
# ============================================================
# SUPPLEMENTARY FIGURES S1 and S2
# S1: pChEMBL distributions by compound group
# S2: Tanimoto distributions by NP classification criterion
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

COLORS = {
    'synthetic' : '#4472C4',
    'coconut'   : '#2E8B57',
    'npscore'   : '#D4A017'
}

# ---- FIGURE S1: pChEMBL distributions ----
fig_s1, axes_s1 = plt.subplots(1, 3, figsize=(15, 5))
fig_s1.suptitle(
    'Figure S1. pChEMBL Value Distributions by Compound Group per Target\n'
    'Red dashed line: activity threshold (pChEMBL = 6.0). '
    'Vertical dash-dot lines: group medians.',
    fontsize=11, fontweight='bold'
)

for idx, target in enumerate(CONFIG['targets']):
    ax = axes_s1[idx]
    combined_df = data[target]['combined_df'].reset_index(drop=True)
    syn_df  = data[target]['syn_df']
    np_df   = data[target]['np_df']

    syn_p     = syn_df['pchembl_value'].dropna()
    coc_p     = np_df[np_df['in_coconut'] == True]['pchembl_value'].dropna()
    nps_p     = np_df[np_df['in_coconut'] == False]['pchembl_value'].dropna()

    bins = np.linspace(5.0, 10.5, 28)

    ax.hist(syn_p, bins=bins, alpha=0.45, density=True,
            color=COLORS['synthetic'], label=f'Synthetic (n={len(syn_p)})')
    ax.hist(coc_p, bins=bins, alpha=0.70, density=True,
            color=COLORS['coconut'],
            label=f'COCONUT-confirmed (n={len(coc_p)})')
    ax.hist(nps_p, bins=bins, alpha=0.55, density=True,
            color=COLORS['npscore'],
            label=f'NP-score-only (n={len(nps_p)})',
            edgecolor='#8B6914', linewidth=0.4)

    ax.axvline(6.0, color='red', linestyle='--',
               linewidth=1.8, label='Activity threshold (6.0)', zorder=5)
    ax.axvline(syn_p.median(), color=COLORS['synthetic'],
               linestyle='-.', linewidth=1.2, alpha=0.9)
    ax.axvline(coc_p.median(), color=COLORS['coconut'],
               linestyle='-.', linewidth=1.2, alpha=0.9)
    ax.axvline(nps_p.median(), color=COLORS['npscore'],
               linestyle='-.', linewidth=1.2, alpha=0.9)

    ax.set_title(
        f'{target}\nSyn median={syn_p.median():.2f}  '
        f'COC median={coc_p.median():.2f}  '
        f'NPS median={nps_p.median():.2f}',
        fontsize=9
    )
    ax.set_xlabel('pChEMBL Value', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.set_xlim(4.8, 11.0)
    ax.legend(fontsize=7, loc='upper right')

plt.tight_layout()
s1_path = Path(CONFIG['output_dir']) / 'FigureS1_pChEMBL_distributions.png'
fig_s1.savefig(str(s1_path), dpi=300, bbox_inches='tight')
plt.show()
print(f'Figure S1 saved: {s1_path.name}')


# ---- FIGURE S2: Tanimoto distributions by criterion ----
fig_s2, axes_s2 = plt.subplots(1, 3, figsize=(15, 5))
fig_s2.suptitle(
    'Figure S2. Maximum Tanimoto Similarity to Synthetic Training Set '
    'by NP Classification Criterion\n'
    'Mann-Whitney U p-values shown for between-group comparison.',
    fontsize=11, fontweight='bold'
)

from scipy import stats

for idx, target in enumerate(CONFIG['targets']):
    ax = axes_s2[idx]
    np_df  = data[target]['np_df'].reset_index(drop=True)
    max_tan = tanimoto_results[target]

    coconut_mask = (np_df['in_coconut'] == True).values
    npscore_mask = (np_df['in_coconut'] == False).values

    tan_coc = max_tan[coconut_mask]
    tan_nps = max_tan[npscore_mask]

    u, p_val = stats.mannwhitneyu(tan_coc, tan_nps, alternative='two-sided')
    p_str = f'p = {p_val:.3f}' if p_val >= 0.001 else 'p < 0.001'

    bins = np.linspace(0.0, 1.0, 26)

    ax.hist(tan_coc, bins=bins, alpha=0.70, density=True,
            color=COLORS['coconut'],
            label=f'COCONUT-confirmed\nn={len(tan_coc)}, '
                  f'median={np.median(tan_coc):.3f}')
    ax.hist(tan_nps, bins=bins, alpha=0.55, density=True,
            color=COLORS['npscore'],
            label=f'NP-score-only\nn={len(tan_nps)}, '
                  f'median={np.median(tan_nps):.3f}',
            edgecolor='#8B6914', linewidth=0.4)

    ax.axvline(np.median(tan_coc), color=COLORS['coconut'],
               linestyle='-.', linewidth=1.5, alpha=0.9)
    ax.axvline(np.median(tan_nps), color=COLORS['npscore'],
               linestyle='-.', linewidth=1.5, alpha=0.9)
    ax.axvline(0.4, color='red', linestyle='--',
               linewidth=1.5, alpha=0.7, label='AD threshold (0.4)')

    equidist_str = 'Equidistant' if p_val > 0.05 else 'Significantly different'
    ax.set_title(
        f'{target}\n{equidist_str}: {p_str}',
        fontsize=10,
        color='#1a5276' if p_val > 0.05 else '#922b21'
    )
    ax.set_xlabel('Max Tanimoto Similarity to Training Set', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.set_xlim(-0.02, 1.02)
    ax.legend(fontsize=7.5)

plt.tight_layout()
s2_path = Path(CONFIG['output_dir']) / 'FigureS2_Tanimoto_by_criterion.png'
fig_s2.savefig(str(s2_path), dpi=300, bbox_inches='tight')
plt.show()
print(f'Figure S2 saved: {s2_path.name}')
print('\n[OK] Both supplementary figures complete.')

## Cell 12: Final Export and Reproducibility Report

In [ ]:
# ============================================================
# CELL 12: Final Export and Reproducibility Report
#
# Exports all results tables, figures, and a complete
# reproducibility report for the paper's supplementary materials.
# ============================================================

import platform
import sklearn

print('Exporting final results...')

# ---- Full results table ----
final_results = results_df.round(4)
final_results.to_csv(
    Path(CONFIG['output_dir']) / 'Table1_full_results.csv',
    index=False
)

# ---- Summary table: mean performance per split strategy ----
summary_pivot = results_df.pivot_table(
    values='MCC',
    index=['Target', 'Model'],
    columns='Split',
    aggfunc='mean'
).round(3)

summary_pivot['degradation'] = (
    summary_pivot.get('random', 0) -
    summary_pivot.get('np_holdout', 0)
).round(3)

summary_pivot.to_csv(
    Path(CONFIG['output_dir']) / 'Table2_mcc_summary_pivot.csv'
)

print('\nMCC Summary (random vs NP holdout degradation):')
print(summary_pivot.to_string())

# ---- Correlation results ----
if 'corr_df' in dir():
    corr_df.to_csv(
        Path(CONFIG['output_dir']) / 'Table3_correlation_results.csv',
        index=False
    )

# ---- Reproducibility report ----
report_path = Path(CONFIG['output_dir']) / 'reproducibility_report.txt'
with open(report_path, 'w') as f:
    f.write('REPRODUCIBILITY REPORT\n')
    f.write('=' * 60 + '\n')
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")

    f.write('ENVIRONMENT\n')
    f.write('-' * 40 + '\n')
    f.write(f'Python: {sys.version}\n')
    f.write(f'Platform: {platform.platform()}\n')

    import importlib
    key_packages = [
        'rdkit', 'pandas', 'numpy', 'sklearn',
        'xgboost', 'torch', 'umap', 'shap'
    ]
    f.write('\nPackage versions:\n')
    for pkg in key_packages:
        try:
            mod = importlib.import_module(pkg if pkg != 'sklearn'
                                          else 'sklearn')
            ver = getattr(mod, '__version__', 'unknown')
            f.write(f'  {pkg}: {ver}\n')
        except ImportError:
            f.write(f'  {pkg}: not available\n')

    f.write('\nCONFIGURATION\n')
    f.write('-' * 40 + '\n')
    for k, v in CONFIG.items():
        f.write(f'  {k}: {v}\n')

    f.write('\nDATASET SUMMARY\n')
    f.write('-' * 40 + '\n')
    for target in CONFIG['targets']:
        if target in data:
            syn_n = len(data[target]['syn_df'])
            np_n = len(data[target]['np_df'])
            f.write(f'  {target}: synthetic={syn_n}, NP={np_n}\n')

print(f'\nReproducibility report saved: {report_path.name}')

# ---- List all output files ----
print('\nAll output files:')
print('=' * 60)
for fpath in sorted(Path(CONFIG['output_dir']).rglob('*')):
    if fpath.is_file():
        size_kb = fpath.stat().st_size / 1024
        print(f'  {fpath.name:50s} {size_kb:8.1f} KB')

print('\n' + '=' * 60)
print('NOTEBOOK 02 COMPLETE')
print('All figures and results tables are in Google Drive:')
print(f"  {CONFIG['output_dir']}")
print('=' * 60)

---

## End of Notebook 02

### Publication figures produced

| Figure | File | Description |
|--------|------|-------------|
| Figure 1 | `Figure1_UMAP_chemical_space.png` | UMAP chemical space: synthetic vs NP compounds |
| Figure 2 | `Figure2_Tanimoto_distributions.png` | Tanimoto similarity distributions (full NP test set) |
| Figure 3 | `Figure3_performance_heatmap.png` | MCC heatmap across models and split strategies |
| Figure 4 | `Figure4_domain_shift_correlation.png` | Performance vs Tanimoto correlation |
| Figure 5 | `Figure5_SHAP_comparison.png` | SHAP feature importance: synthetic vs NP test |
| Figure S1 | `FigureS1_pChEMBL_distributions.png` | pChEMBL distributions by compound group |
| Figure S2 | `FigureS2_Tanimoto_by_criterion.png` | Tanimoto distributions by NP classification criterion |

### Results tables produced

| File | Description |
|------|-------------|
| `Table1_full_results.csv` | All 36 model evaluations (4 models x 3 splits x 3 targets) |
| `Table2_mcc_summary_pivot.csv` | MCC pivot with degradation scores |
| `Table3_correlation_results.csv` | Spearman correlations (Tanimoto vs performance outcomes) |
| `multiseed_results.csv` | Multi-seed stability evaluation (5 seeds, Random Forest) |
| `sensitivity_coconut_only.csv` | Subgroup performance: COCONUT-confirmed vs NP-score-only |
| `rebalance_results.csv` | Activity-rebalanced COCONUT evaluation results |
| `tanimoto_by_criterion.csv` | Tanimoto similarity statistics by NP classification criterion |
| `reproducibility_report.txt` | Full environment and configuration log |

### Repository structure
```
np_qsar_project/
├── 01_data_curation.ipynb       # Notebook 01: ChEMBL + COCONUT curation
├── 02_modeling_pipeline.ipynb   # This notebook
├── data/                        # Curated CSVs (generated by Notebook 01)
│   ├── AChE_synthetic_train.csv
│   ├── AChE_np_test.csv
│   ├── AChE_combined.csv
│   └── ... (same pattern for 5LOX, COX2)
└── results/                     # Figures and tables (generated by this notebook)
    ├── Figure1_UMAP_chemical_space.png
    ├── Figure2_Tanimoto_distributions.png
    └── ...
```

### Reproducibility
All random seeds are fixed at 42 throughout. Multi-seed stability confirmed for Random Forest
(MCC SD ≤ 0.038 across seeds 42, 7, 13, 99, 2024). Full environment details are saved to
`reproducibility_report.txt` on completion.

### License
Code released under MIT License. Data derived from ChEMBL (CC BY-SA 3.0) and COCONUT 2.0 (CC0).
See repository README for full license details.